<a href="https://www.kaggle.com/code/avikdas567/multi-view-pig-posture-recognition?scriptVersionId=311145892" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os, cv2, ast, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# =========================
# CONFIG
# =========================
class CFG:
    DATA_PATH = "/kaggle/input/competitions/multi-view-pig-posture-recognition/multiview_pig_posture_recognition"
    
    IMG_SIZE = 128
    BATCH_SIZE = 128
    EPOCHS = 2
    
    MODEL_NAME = "tf_efficientnet_b0"
    LR = 1e-3
    
    DEVICE = "cuda"
    NUM_WORKERS = 2
    
    SEED = 42

torch.backends.cudnn.benchmark = True

# =========================
# Seed
# =========================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)

# =========================
# DATA
# =========================
train_df = pd.read_csv(f"{CFG.DATA_PATH}/train2.csv")
test_df = pd.read_csv(f"{CFG.DATA_PATH}/test.csv")

train_df["bbox"] = train_df["bbox"].apply(ast.literal_eval)
test_df["bbox"] = test_df["bbox"].apply(ast.literal_eval)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df["class_id"],
    random_state=CFG.SEED
)

print("Train:", len(train_df), "Val:", len(val_df))

# =========================
# DATASET
# =========================
class PigDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.loc[idx]
        
        img_path = os.path.join(
            CFG.DATA_PATH,
            "test_images" if self.is_test else "train2_images",
            row["image_id"]
        )
        
        img = cv2.imread(img_path)
        
        x, y, w, h = map(int, row["bbox"])
        h_img, w_img, _ = img.shape
        
        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(w_img, x + w)
        y2 = min(h_img, y + h)
        
        crop = img[y1:y2, x1:x2]
        
        crop = cv2.resize(crop, (CFG.IMG_SIZE, CFG.IMG_SIZE))
        crop = crop.transpose(2, 0, 1)
        crop = torch.from_numpy(crop).float() / 255.0
        
        if self.is_test:
            return crop, row["row_id"]
        
        return crop, row["class_id"]

# =========================
# DATALOADER
# =========================
train_loader = DataLoader(
    PigDataset(train_df),
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    PigDataset(val_df),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    PigDataset(test_df, is_test=True),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

# =========================
# MODEL
# =========================
model = timm.create_model(
    CFG.MODEL_NAME,
    pretrained=True,
    num_classes=5
).to(CFG.DEVICE)

# =========================
# LOSS
# =========================
class_counts = train_df["class_id"].value_counts().sort_index().values
weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(CFG.DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR)
scaler = GradScaler()

# =========================
# TRAIN
# =========================
best_loss = np.inf

for epoch in range(CFG.EPOCHS):
    print(f"\n===== EPOCH {epoch+1} =====")
    
    model.train()
    running_loss = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images = images.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        
        optimizer.zero_grad()
        
        with autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    print(f"Loss: {epoch_loss:.4f}")
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "best.pth")
        print("Saved Best Model.")

# =========================
# INFERENCE
# =========================
model.load_state_dict(torch.load("best.pth"))
model.eval()

row_ids, preds = [], []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Inference"):
        images = images.to(CFG.DEVICE, non_blocking=True)
        outputs = model(images)
        pred = outputs.argmax(1).cpu().numpy()
        
        preds.extend(pred)
        row_ids.extend(ids)

sub = pd.DataFrame({"row_id": row_ids, "class_id": preds})
sub.to_csv("submission.csv", index=False)

print("DONE. 'submission.csv' created")

Train: 21105 Val: 2345


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]


===== EPOCH 1 =====


Training: 100%|██████████| 165/165 [02:54<00:00,  1.06s/it]


Loss: 0.8152
Saved Best Model.

===== EPOCH 2 =====


Training: 100%|██████████| 165/165 [01:47<00:00,  1.54it/s]


Loss: 0.2707
Saved Best Model.


Inference: 100%|██████████| 92/92 [00:42<00:00,  2.15it/s]

DONE. 'submission.csv' created
